# Panopticon Protocol v3 - Research Plot Generation for Colab

This notebook regenerates the publication-style training figures and the evaluation/reward diagnostics used in the README.

It is designed for the finished 50-episode A10G worker run. Upload the saved `output_logs.txt` from the worker bucket or model repo, then fetch the latest plotting scripts directly from GitHub so the notebook stays in sync with the repository.


## Usage

1. Upload `output_logs.txt` (or `output_log.txt`) from the finished worker run.
2. Fetch the latest `generate_plots.py` from the GitHub repo.
3. Run the execution cell to create all training figures inside `plots/`.
4. Inspect `plots/training_statistics.md` and the preview cells.
5. Upload `evaluationResults.json` (or `evaluation_results.json` / `evaluation_snapshot_apr26.json`) for reward and benchmark diagnostics.
6. Run the evaluation plot cells to generate the reward-focused figure suite inside `plots/`.
7. Use the preview cells to inspect the outputs directly in Colab.


In [ ]:
!pip install -q matplotlib numpy

In [ ]:
from pathlib import Path

try:
    from google.colab import files  # type: ignore
except ImportError:
    files = None

candidates = [Path('output_logs.txt'), Path('output_log.txt')]
if files is not None and not any(path.exists() for path in candidates):
    print('Upload your training log (`output_logs.txt` or `output_log.txt`).')
    files.upload()

existing = [path for path in candidates if path.exists()]
if not existing:
    raise FileNotFoundError('No training log found. Upload `output_logs.txt` or `output_log.txt`.')

print(f'Using log file: {existing[0]}')

In [ ]:
from pathlib import Path
from urllib.request import urlopen

repo_root = 'https://raw.githubusercontent.com/Ayush-Kumar0207/panopticon-protocol-v3/main'
script_url = f'{repo_root}/generate_plots.py'
script_text = urlopen(script_url).read().decode('utf-8')
Path('generate_plots.py').write_text(script_text, encoding='utf-8')
print('Fetched generate_plots.py from the repo.')


In [ ]:
!python generate_plots.py

In [ ]:
from IPython.display import Image, Markdown, display

preview_files = [
    'curriculum_loss_overview.png',
    'per_level_convergence.png',
    'expert_reward_progression.png',
    'expert_grade_distribution.png',
    'expert_operational_metrics.png',
    'optimization_diagnostics.png',
    'dataset_scaling.png',
    'curriculum_heatmap.png',
]

for filename in preview_files:
    path = Path('plots') / filename
    if path.exists():
        display(Markdown(f'### {filename}'))
        display(Image(filename=str(path)))


In [ ]:
stats_path = Path('plots') / 'training_statistics.md'
if stats_path.exists():
    display(Markdown(stats_path.read_text(encoding='utf-8')))
else:
    print('Run the plot generation cell first.')

## Evaluation + Reward Diagnostics

Upload the benchmark payload generated by `full_evaluation.py` (`evaluationResults.json` / `evaluation_results.json`) or the historical repo snapshot `evaluation_snapshot_apr26.json`. The next cells fetch the latest `generate_evaluation_plots.py` from the repo, regenerate the reward-focused figures into `plots/`, and preview them inside Colab.


In [ ]:
from pathlib import Path

try:
    from google.colab import files  # type: ignore
except ImportError:
    files = None

evaluation_candidates = [Path('evaluationResults.json'), Path('evaluation_results.json'), Path('evaluation_snapshot_apr26.json')]
if files is not None and not any(path.exists() for path in evaluation_candidates):
    print('Upload your evaluation payload (`evaluationResults.json`, `evaluation_results.json`, or `evaluation_snapshot_apr26.json`).')
    files.upload()

evaluation_existing = [path for path in evaluation_candidates if path.exists()]
if not evaluation_existing:
    raise FileNotFoundError('No evaluation payload found. Upload `evaluationResults.json`, `evaluation_results.json`, or `evaluation_snapshot_apr26.json`.')

evaluation_input = str(evaluation_existing[0])
print(f'Using evaluation payload: {evaluation_input}')

In [ ]:
from pathlib import Path
from urllib.request import urlopen

repo_root = 'https://raw.githubusercontent.com/Ayush-Kumar0207/panopticon-protocol-v3/main'
script_url = f'{repo_root}/generate_evaluation_plots.py'
script_text = urlopen(script_url).read().decode('utf-8')
Path('generate_evaluation_plots.py').write_text(script_text, encoding='utf-8')
print('Fetched generate_evaluation_plots.py from the repo.')

In [ ]:
!python generate_evaluation_plots.py --input {evaluation_input} --plot-dir plots --timeline-level level_5

In [ ]:
from pathlib import Path
from IPython.display import Image, Markdown, display

evaluation_preview_files = [
    'comparison_grades.png',
    'comparison_operations.png',
    'comparison_radar.png',
    'reward_distributions.png',
    'reward_frontier.png',
    'reward_turn_dynamics.png',
    'scenario_timeline.png',
]

for filename in evaluation_preview_files:
    path = Path('plots') / filename
    if path.exists():
        display(Markdown(f'### {filename}'))
        display(Image(filename=str(path)))

In [ ]:
import json
from pathlib import Path
from IPython.display import Markdown, display

payload = json.loads(Path(evaluation_input).read_text(encoding='utf-8'))
rows = payload.get('comparison_rows', [])
if not rows:
    print('No comparison rows found in the evaluation payload.')
else:
    markdown_lines = [
        '### Evaluation Summary',
        '',
        '| Agent | Level | Grade Mean | Reward Mean | Revenue Mean | Security Mean | Sleepers Caught |',
        '|---|---|---:|---:|---:|---:|---:|',
    ]
    for row in rows:
        markdown_lines.append(
            f"| {row['agent']} | {row['level']} | {row['grade_mean']:.3f} | {row['reward_mean']:.2f} | {row['revenue_mean']:.1f} | {row['security_mean']:.1f} | {row['sleepers_caught_mean']:.2f} |"
        )
    display(Markdown('\n'.join(markdown_lines)))